# NB05 · 失败归因

| | |
|---|---|
| **目标** | 给 NB02 policy 的失败建立 taxonomy，回答采数据的黄金问题：「只能采 20 条新数据，采什么？」 |
| **前置** | NB02 的 checkpoint |
| **预计耗时** | 1 天（其中 2–3 小时是人肉看视频——不许跳） |
| **产出物** | `results/NB05_labels.csv`（人肉标注）+ 失败分布图 |
| **通过标准** | 30 条 rollout 全部标注完毕，且能说出 top-3 失败模式的机理 |

规则：从上到下顺序执行；每个 ✅ 检查点必须核对；最后的复盘必须填写并 commit。


In [ ]:
from pathlib import Path
import pandas as pd
import nbutils

N_ROLLOUTS = 30
VIDEO_DIR = Path("outputs/nb05/videos"); VIDEO_DIR.mkdir(parents=True, exist_ok=True)
LABELS_CSV = Path("results/NB05_labels.csv")

In [ ]:
# 采集失败素材：优先用官方 eval 脚本的存视频功能（--help 里找 video/save 相关参数），
# 例如: python -m lerobot.scripts.eval --policy.path=<CKPT> --env.type=pusht \
#          --eval.n_episodes=30 --output_dir=outputs/nb05 (+ 视频开关)
# 跑完把每条的 success 与视频路径整理成标注表：
import glob
videos = sorted(glob.glob("outputs/nb05/**/*.mp4", recursive=True))
print(f"found {len(videos)} videos")
if not LABELS_CSV.exists():
    pd.DataFrame({
        "episode": range(len(videos)),
        "video": videos,
        "success": None,        # 从 eval 输出填
        "label": None,          # 失败的填 taxonomy 标签；成功留空
        "note": None,           # 一句话：失败发生在哪一秒、什么样子
    }).to_csv(LABELS_CSV, index=False)
    print(f"标注表已生成: {LABELS_CSV}")

### Taxonomy 定义卡（判定规则各一句，有歧义时按第一个命中的算）

| 标签 | 判定规则 |
|---|---|
| `perception` | 状态/图像里关键信息缺失或错误（遮挡、出视野），policy 输入已经错了 |
| `policy` | 输入正常，但输出动作系统性错误（推错方向、绕圈、停滞震荡） |
| `execution` | 决策合理，物理执行走样（打滑、过冲、接触动力学出乎意料） |
| `env` | 初始状态本身超出训练分布（T 块在数据集中从未出现的位置） |
| `protocol` | 失败是评测协议造成的（超时截断了本会成功的轨迹、success 阈值边缘） |


In [ ]:
# 逐条看视频（notebook 内嵌播放），边看边填 CSV——用任何编辑器直接改 NB05_labels.csv
from IPython.display import Video, display
df = pd.read_csv(LABELS_CSV)
todo = df[df["label"].isna() & (df["success"] == False)]
print(f"待标注失败条数: {len(todo)}")
if len(todo):
    row = todo.iloc[0]
    print(f"episode {row['episode']}: {row['video']}")
    display(Video(row["video"], embed=True, width=480))

In [ ]:
# 全部标完后：失败分布
df = pd.read_csv(LABELS_CSV)
assert df[df["success"] == False]["label"].notna().all(), "还有未标注的失败条目"
dist = df[df["success"] == False]["label"].value_counts()
ax = dist.plot.bar(figsize=(6, 3), title="failure distribution")
ax.figure.savefig("results/NB05_dist.png", dpi=120, bbox_inches="tight")
sr = float(df["success"].mean())
print(f"success = {sr:.1%}")
nbutils.log_result("NB05", {"n": len(df), "success_rate": sr, "failure_dist": dist.to_dict()})

## 分析

1. **Top-3 失败逐帧描述**（每条一段）：失败在第几秒、触发条件、机理假设、什么数据/改动能修。
2. **黄金问题**：只能采 20 条新数据来提升 success rate——初始状态放哪、演示什么行为（含恢复动作吗）、为什么是这 20 条而不是别的？要求引用你的分布图作为证据。
3. **对照你的金融直觉**：这就是 PnL 归因——失败分布是你的因子暴露，采数据是调仓。写一段两者的映射（这段话未来面试直接用）。
4. **协议自查**：有没有失败其实该归 `protocol`？如果有，先修评测协议再谈模型——这是评测专家和调参侠的又一分界线。


## 复盘（必填，不填不算完成这本 notebook）

> 复盘写在这里并 commit。允许粗糙，禁止事后美化。

- **预期 vs 实际**：
- **最大的一个意外**：
- **卡最久的一步和根因**：
- **用一句话向非技术人解释本次学到的东西**：
- **进入下一本之前要做的一个动作**：
